In [2]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week8-assignment-5"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

## Lead and Lag

In [3]:
windowDF = spark.read \
.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/public/trendytech/datasets/windowdatamodified.csv")

In [4]:
from pyspark.sql import Window

In [5]:
mywindow = Window.partitionBy("country").orderBy("weeknum")

### add a column referring to previous row using lag().over()

In [11]:
windowDF1 = windowDF.withColumn("prev_week",lag("invoicevalue").over(mywindow))

In [18]:
windowDF2 = windowDF1.withColumn("diff_week", expr("invoicevalue - prev_week"))  #finding how each week performed compared to prev week

In [13]:
windowDF2.show()

+-------+-------+-----------+-------------+------------+---------+-------------------+
|country|weeknum|numinvoices|totalquantity|invoicevalue|prev_week|          diff_week|
+-------+-------+-----------+-------------+------------+---------+-------------------+
| Sweden|     50|          3|         3714|      2646.3|     null|               null|
|Germany|     48|         11|         1795|      1600.0|     null|               null|
|Germany|     49|         12|         1852|      1800.0|   1600.0|              200.0|
|Germany|     50|         15|         1973|      1800.0|   1800.0|                0.0|
|Germany|     51|          5|         1103|      1600.0|   1800.0|             -200.0|
| France|     48|          4|         1299|       500.0|     null|               null|
| France|     49|          9|         2303|       500.0|    500.0|                0.0|
| France|     50|          6|          529|      537.32|    500.0|  37.32000000000005|
| France|     51|          5|          847|

### add a column referring to next row using lead().over()

In [14]:
windowDF3 = windowDF.withColumn("next_week",lead("invoicevalue").over(mywindow))

In [19]:
windowDF4 = windowDF3.withColumn("diff_week", expr(" next_week - invoicevalue"))  #finding how each week performed compared to next week

In [17]:
windowDF4.show()

+-------+-------+-----------+-------------+------------+---------+-------------------+
|country|weeknum|numinvoices|totalquantity|invoicevalue|next_week|          diff_week|
+-------+-------+-----------+-------------+------------+---------+-------------------+
| Sweden|     50|          3|         3714|      2646.3|     null|               null|
|Germany|     48|         11|         1795|      1600.0|   1800.0|              200.0|
|Germany|     49|         12|         1852|      1800.0|   1800.0|                0.0|
|Germany|     50|         15|         1973|      1800.0|   1600.0|             -200.0|
|Germany|     51|          5|         1103|      1600.0|     null|               null|
| France|     48|          4|         1299|       500.0|    500.0|                0.0|
| France|     49|          9|         2303|       500.0|   537.32|  37.32000000000005|
| France|     50|          6|          529|      537.32|    500.0| -37.32000000000005|
| France|     51|          5|          847|